<a href="https://colab.research.google.com/github/Vishu235/MetaBEARS/blob/main/colab/MetaBEARS_MiniKandinsky.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MetaBEARS - MiniKandinsky Training and Evaluation

This notebook trains reproducible MiniKandinsky ensembles and evaluates them with MetaBEARS diagnostics, controlled semantic interventions, review-budget calibration, and a held-out OOD shift.

## Before running

1. Select **Runtime -> Change runtime type -> T4 GPU**.
2. Upload `kand-3k.zip` to:

`MyDrive/PES - Semester 4/metabears_data/kand-3k.zip`

The local source archive is available under the Phase I `bears/XOR_MNIST/data` folder. Existing checkpoints under `metabears_minikandinsky/seed_*` can be evaluated without retraining.


In [ ]:
#@title 1. Configuration
REPO_URL = "https://github.com/Vishu235/MetaBEARS.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/MetaBEARS"  #@param {type:"string"}

DRIVE_DATA_DIR = "/content/drive/MyDrive/PES - Semester 4/metabears_data"  #@param {type:"string"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/PES - Semester 4/metabears_minikandinsky"  #@param {type:"string"}
KAND_ZIP = f"{DRIVE_DATA_DIR}/kand-3k.zip"

RUN_SMOKE = False  #@param {type:"boolean"}
RUN_FULL_TRAINING = False  #@param {type:"boolean"}
RUN_METABEARS_SMOKE = False  #@param {type:"boolean"}
RUN_METABEARS_FULL = False  #@param {type:"boolean"}
RUN_V1_TRAINING = False  #@param {type:"boolean"}
RUN_V1_CSUP1_EVALUATION = False  #@param {type:"boolean"}
RUN_V1_CSUP0_EVALUATION = False  #@param {type:"boolean"}
RUN_V2_ENTROPY_TRAINING = False  #@param {type:"boolean"}
RUN_V2_REPRESENTATION_SWEEP = False  #@param {type:"boolean"}
RUN_V3_SCORING_SWEEP = False  #@param {type:"boolean"}
RUN_V4_UNCERTAINTY_ABLATION = False  #@param {type:"boolean"}
SMOKE_EPOCHS = 1  #@param {type:"integer"}
FULL_EPOCHS = 30  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
SEEDS = [0, 10, 20]

print("Configuration ready.")
print("Full training enabled:", RUN_FULL_TRAINING)
print("MetaBEARS smoke/full:", RUN_METABEARS_SMOKE, RUN_METABEARS_FULL)
print("v1 training:", RUN_V1_TRAINING)
print("v1 c_sup=1/c_sup=0 evaluation:", RUN_V1_CSUP1_EVALUATION, RUN_V1_CSUP0_EVALUATION)
print("v2 entropy training/sweep:", RUN_V2_ENTROPY_TRAINING, RUN_V2_REPRESENTATION_SWEEP)
print("v3 scoring sweep:", RUN_V3_SCORING_SWEEP)
print("v4 uncertainty ablation:", RUN_V4_UNCERTAINTY_ABLATION)


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Clone or update MetaBEARS
import os
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
if repo.exists() and (repo / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin'], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=repo, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=repo, check=True)
elif repo.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)


In [ ]:
#@title 4. Install Colab-safe dependencies
import subprocess

subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.colab.txt'], cwd=REPO_DIR, check=True)
print('Dependencies installed without replacing Colab CUDA PyTorch.')


In [ ]:
#@title 5. Runtime diagnostics
import subprocess
subprocess.run(['python', 'colab_runner.py', '--job', 'diagnostics'], cwd=REPO_DIR, check=True)


In [ ]:
#@title 6. Validate and stage kand-3k.zip
import hashlib
import shutil
import zipfile
from pathlib import Path

source_zip = Path(KAND_ZIP)
if not source_zip.exists():
    raise FileNotFoundError(f'Upload kand-3k.zip to {source_zip}')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

with zipfile.ZipFile(source_zip) as archive:
    corrupt = archive.testzip()
    if corrupt is not None:
        raise RuntimeError(f'Corrupt member in kand-3k.zip: {corrupt}')

repo_zip = Path(REPO_DIR) / 'XOR_MNIST' / 'data' / 'kand-3k.zip'
repo_zip.parent.mkdir(parents=True, exist_ok=True)
if not repo_zip.exists() or sha256(repo_zip) != sha256(source_zip):
    shutil.copy2(source_zip, repo_zip)

Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print('Dataset ready:', repo_zip)
print('SHA-256:', sha256(repo_zip))


In [ ]:
#@title 7. Reusable job helper
import shlex
import subprocess
from pathlib import Path

def run_job(job, *, seed, epochs, batch_size, c_sup=1.0, w_c=10.0, checkpoint_tag=None, entropy=False, w_h=1.0):
    command = [
        'python', 'colab_runner.py', '--job', job,
        '--seed', str(seed), '--epochs', str(epochs),
        '--batch-size', str(batch_size),
        '--minikand-c-sup', str(c_sup), '--minikand-w-c', str(w_c),
        '--minikand-w-h', str(w_h),
    ]
    if checkpoint_tag is not None:
        command.extend(['--minikand-checkpoint-tag', checkpoint_tag])
    if entropy:
        command.append('--minikand-entropy')
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    completed = subprocess.run(command, cwd=REPO_DIR, check=False)
    if completed.returncode:
        logs = sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda p: p.stat().st_mtime)
        if logs:
            print(''.join(logs[-1].read_text(errors='replace').splitlines(True)[-160:]))
        raise SystemExit(f'{job} failed with exit code {completed.returncode}')
    return sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda p: p.stat().st_mtime)[-1]

print('Job helper ready.')


In [ ]:
#@title 8. One-epoch MiniKandinsky smoke test
if RUN_SMOKE:
    smoke_log = run_job('minikand_smoke', seed=0, epochs=SMOKE_EPOCHS, batch_size=BATCH_SIZE)
    smoke_target = Path(DRIVE_RESULTS_DIR) / 'smoke' / smoke_log.name
    smoke_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(smoke_log, smoke_target)
    print('Smoke test passed. Log saved to:', smoke_target)
else:
    print('Smoke test disabled.')


In [ ]:
#@title 9. Three-seed MiniKandinsky baseline training
import json
from datetime import datetime, timezone

if RUN_FULL_TRAINING:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    run_records = []
    for seed in SEEDS:
        log_path = run_job('minikand_train', seed=seed, epochs=FULL_EPOCHS, batch_size=BATCH_SIZE)
        checkpoint = Path(REPO_DIR) / 'XOR_MNIST' / 'data' / 'ckpts' / f'minikandinsky-minikanddpl-dis-{seed}-end.pt'
        if not checkpoint.exists():
            raise FileNotFoundError(checkpoint)
        seed_dir = Path(DRIVE_RESULTS_DIR) / f'seed_{seed}'
        seed_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(checkpoint, seed_dir / checkpoint.name)
        shutil.copy2(log_path, seed_dir / log_path.name)
        run_records.append({'seed': seed, 'checkpoint': checkpoint.name, 'log': log_path.name})

    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': commit, 'dataset': 'minikandinsky',
        'model': 'minikanddpl', 'task': 'mini_patterns_bombazza',
        'epochs': FULL_EPOCHS, 'batch_size': BATCH_SIZE,
        'seeds': SEEDS, 'dataset_sha256': sha256(Path(KAND_ZIP)),
        'runs': run_records,
    }
    manifest_path = Path(DRIVE_RESULTS_DIR) / 'baseline_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print('Training complete. Manifest:', manifest_path)
else:
    print('Full training disabled. Enable it only after the smoke test passes.')


In [ ]:
#@title 10. Resolve the trained ensemble checkpoints
CHECKPOINT_PATHS = [
    str(Path(DRIVE_RESULTS_DIR) / f'seed_{seed}' / f'minikandinsky-minikanddpl-dis-{seed}-end.pt')
    for seed in SEEDS
]
if RUN_METABEARS_SMOKE or RUN_METABEARS_FULL:
    missing = [path for path in CHECKPOINT_PATHS if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Missing trained checkpoints: ' + ', '.join(missing))
    print('Evaluation ensemble:')
    for checkpoint in CHECKPOINT_PATHS:
        print(' -', checkpoint)
else:
    print('MetaBEARS evaluation disabled in the Configuration cell.')


In [ ]:
#@title 11. Reusable MetaBEARS evaluation helper
import time

def run_metabears_evaluation(intervention, ood_transform, *, smoke=False):
    mode = 'smoke' if smoke else 'full'
    run_name = f"{time.strftime('%Y%m%d_%H%M%S')}_{intervention}_{mode}"
    output_dir = Path(DRIVE_RESULTS_DIR) / 'evaluations' / run_name
    command = [
        'python', 'colab_runner.py', '--job', 'metabears_minikandinsky',
        '--seed', '0', '--batch-size', str(BATCH_SIZE),
        '--minikand-output-dir', str(output_dir),
        '--minikand-intervention', intervention,
        '--minikand-ood-transform', ood_transform,
        '--minikand-checkpoints', *CHECKPOINT_PATHS,
    ]
    if smoke:
        command.extend(['--metabears-max-batches', '1'])
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    completed = subprocess.run(command, cwd=REPO_DIR, check=False)
    logs = sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda path: path.stat().st_mtime)
    if completed.returncode:
        if logs:
            print(''.join(logs[-1].read_text(errors='replace').splitlines(True)[-200:]))
        raise SystemExit(f'MetaBEARS evaluation failed with exit code {completed.returncode}')
    summary_path = output_dir / 'run_summary.json'
    if not summary_path.is_file():
        raise FileNotFoundError(summary_path)
    if logs:
        shutil.copy2(logs[-1], output_dir / logs[-1].name)
    summary = json.loads(summary_path.read_text())
    print('Saved evaluation:', output_dir)
    print('ID task accuracy:', summary['splits']['id_test']['task_accuracy'])
    if summary.get('ood_detection') is not None:
        print('OOD AUROC:', summary['ood_detection']['auroc'])
    return output_dir, summary

print('MetaBEARS helper ready.')


In [ ]:
#@title 12. One-batch MetaBEARS integration check
if RUN_METABEARS_SMOKE:
    smoke_output, smoke_summary = run_metabears_evaluation(
        'figure_permute', 'palette_desaturate', smoke=True
    )
    print('MetaBEARS integration check passed:', smoke_output)
else:
    print('MetaBEARS integration check disabled.')


In [ ]:
#@title 13. Full MiniKandinsky MetaBEARS evaluation
if RUN_METABEARS_FULL:
    evaluation_records = []
    evaluation_matrix = [
        ('figure_permute', 'palette_desaturate'),
        ('palette_cycle', 'none'),
    ]
    for intervention, ood_transform in evaluation_matrix:
        output_dir, summary = run_metabears_evaluation(
            intervention, ood_transform, smoke=False
        )
        evaluation_records.append({
            'intervention': intervention,
            'ood_transform': ood_transform,
            'output_directory': str(output_dir),
            'run_summary': str(output_dir / 'run_summary.json'),
            'id_task_accuracy': summary['splits']['id_test']['task_accuracy'],
            'id_concept_accuracy': summary['splits']['id_test']['concept_accuracy'],
            'ood_auroc': (summary['ood_detection']['auroc'] if summary.get('ood_detection') else None),
        })
    evaluation_manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'dataset_sha256': sha256(Path(KAND_ZIP)),
        'checkpoints': CHECKPOINT_PATHS,
        'runs': evaluation_records,
    }
    manifest_path = Path(DRIVE_RESULTS_DIR) / 'evaluations' / f"evaluation_manifest_{time.strftime('%Y%m%d_%H%M%S')}.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(evaluation_manifest, indent=2))
    print('Full evaluation complete. Manifest:', manifest_path)
else:
    print('Full MetaBEARS evaluation disabled.')


In [ ]:
#@title 14. Train matched v1 supervision-control ensembles
V1_TRAINING_CONDITIONS = (
    ('csup_1', 1.0, 10.0),
    ('csup_0', 0.0, 0.0),
)
if RUN_V1_TRAINING:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    v1_training_records = []
    for condition, c_sup, w_c in V1_TRAINING_CONDITIONS:
        for seed in SEEDS:
            checkpoint_name = (
                'minikandinsky-minikanddpl-v1-task-loss-'
                f'csup-{int(c_sup)}-wc-{int(w_c)}-seed-{seed}-end.pt'
            )
            seed_dir = Path(DRIVE_RESULTS_DIR) / 'training_v1' / condition / f'seed_{seed}'
            drive_checkpoint = seed_dir / checkpoint_name
            if drive_checkpoint.is_file():
                print('Reusing completed v1 checkpoint:', drive_checkpoint)
                v1_training_records.append({
                    'condition': condition, 'seed': seed,
                    'checkpoint': str(drive_checkpoint), 'reused': True,
                })
                continue
            log_path = run_job(
                'minikand_train', seed=seed, epochs=FULL_EPOCHS,
                batch_size=BATCH_SIZE, c_sup=c_sup, w_c=w_c,
                checkpoint_tag='v1-task-loss',
            )
            local_checkpoint = Path(REPO_DIR) / 'XOR_MNIST' / 'data' / 'ckpts' / checkpoint_name
            if not local_checkpoint.is_file():
                raise FileNotFoundError(local_checkpoint)
            seed_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_checkpoint, drive_checkpoint)
            shutil.copy2(log_path, seed_dir / log_path.name)
            v1_training_records.append({
                'condition': condition, 'seed': seed,
                'checkpoint': str(drive_checkpoint),
                'log': log_path.name, 'reused': False,
            })

    v1_training_manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': commit, 'dataset': 'minikandinsky',
        'model': 'minikanddpl', 'task': 'mini_patterns_bombazza',
        'training_objective': 'task NLL plus optional concept cross-entropy',
        'epochs': FULL_EPOCHS, 'batch_size': BATCH_SIZE,
        'seeds': SEEDS, 'dataset_sha256': sha256(Path(KAND_ZIP)),
        'runs': v1_training_records,
    }
    v1_training_manifest_path = Path(DRIVE_RESULTS_DIR) / 'training_v1' / 'training_manifest.json'
    v1_training_manifest_path.parent.mkdir(parents=True, exist_ok=True)
    v1_training_manifest_path.write_text(json.dumps(v1_training_manifest, indent=2))
    print('Matched v1 training complete:', v1_training_manifest_path)
else:
    print('Matched v1 training disabled.')


In [ ]:
#@title 15. Resolve matched v1 checkpoints
def v1_checkpoint_paths(condition, c_sup, w_c):
    return [
        str(Path(DRIVE_RESULTS_DIR) / 'training_v1' / condition / f'seed_{seed}' /
            ('minikandinsky-minikanddpl-v1-task-loss-'
             f'csup-{c_sup}-wc-{w_c}-seed-{seed}-end.pt'))
        for seed in SEEDS
    ]

V1_CSUP1_CHECKPOINT_PATHS = v1_checkpoint_paths('csup_1', 1, 10)
V1_CSUP0_CHECKPOINT_PATHS = v1_checkpoint_paths('csup_0', 0, 0)
for enabled, condition, paths in (
    (RUN_V1_CSUP1_EVALUATION, 'c_sup=1', V1_CSUP1_CHECKPOINT_PATHS),
    (RUN_V1_CSUP0_EVALUATION, 'c_sup=0', V1_CSUP0_CHECKPOINT_PATHS),
):
    if enabled:
        missing = [path for path in paths if not Path(path).is_file()]
        if missing:
            raise FileNotFoundError(f'Missing {condition} v1 checkpoints: ' + ', '.join(missing))
        print(condition, 'v1 evaluation ensemble ready.')


In [ ]:
#@title 16. MetaBEARS v1 held-out-shift evaluation helper
def run_metabears_v1(checkpoints, *, tag, c_sup, w_c, intervention):
    run_name = f"{time.strftime('%Y%m%d_%H%M%S')}_{tag}_{intervention}"
    output_dir = Path(DRIVE_RESULTS_DIR) / 'evaluations_v1' / run_name
    command = [
        'python', 'colab_runner.py', '--job', 'metabears_minikandinsky',
        '--seed', '0', '--batch-size', str(BATCH_SIZE),
        '--minikand-output-dir', str(output_dir),
        '--minikand-c-sup', str(c_sup), '--minikand-w-c', str(w_c),
        '--minikand-intervention', intervention,
        '--minikand-representation-key', 'pCS',
        '--minikand-representation-normalization', 'zscore_l2',
        '--minikand-ood-validation-transform', 'palette_desaturate',
        '--minikand-ood-transform', 'palette_pastel',
        '--minikand-shortcut-max-false-review-rate', '0.05',
        '--minikand-familiarity-max-false-review-rate', '0.05',
        '--minikand-checkpoints', *checkpoints,
    ]
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    completed = subprocess.run(command, cwd=REPO_DIR, check=False)
    logs = sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda path: path.stat().st_mtime)
    if completed.returncode:
        if logs:
            print(''.join(logs[-1].read_text(errors='replace').splitlines(True)[-200:]))
        raise SystemExit(f'MetaBEARS v1 evaluation failed with exit code {completed.returncode}')
    summary_path = output_dir / 'run_summary.json'
    summary = json.loads(summary_path.read_text())
    if logs:
        shutil.copy2(logs[-1], output_dir / logs[-1].name)
    print('Saved v1 evaluation:', output_dir)
    print('ID review rate:', summary['splits']['id_test']['review_rate'])
    print('Held-out OOD AUROC:', summary['ood_detection']['auroc'])
    return output_dir, summary

print('MetaBEARS v1 helper ready.')


In [ ]:
#@title 17. Run the v1 supervision-control matrix
v1_specs = []
if RUN_V1_CSUP1_EVALUATION:
    missing = [path for path in V1_CSUP1_CHECKPOINT_PATHS if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Missing c_sup=1 checkpoints: ' + ', '.join(missing))
    v1_specs.append(('csup_1', V1_CSUP1_CHECKPOINT_PATHS, 1.0, 10.0))
if RUN_V1_CSUP0_EVALUATION:
    v1_specs.append(('csup_0', V1_CSUP0_CHECKPOINT_PATHS, 0.0, 0.0))

if v1_specs:
    v1_records = []
    for tag, checkpoints, c_sup, w_c in v1_specs:
        for intervention in ('figure_permute', 'palette_cycle'):
            output_dir, summary = run_metabears_v1(
                checkpoints, tag=tag, c_sup=c_sup, w_c=w_c,
                intervention=intervention,
            )
            v1_records.append({
                'training_condition': tag, 'intervention': intervention,
                'output_directory': str(output_dir),
                'id_task_accuracy': summary['splits']['id_test']['task_accuracy'],
                'id_concept_accuracy': summary['splits']['id_test']['concept_accuracy'],
                'id_review_rate': summary['splits']['id_test']['review_rate'],
                'heldout_ood_auroc': summary['ood_detection']['auroc'],
                'heldout_ood_average_precision': summary['ood_detection']['average_precision'],
            })
    v1_manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'dataset_sha256': sha256(Path(KAND_ZIP)),
        'protocol': {
            'representation': 'pCS', 'normalization': 'zscore_l2',
            'ood_validation': 'palette_desaturate', 'ood_test': 'palette_pastel',
            'shortcut_false_review_budget': 0.05,
            'familiarity_false_review_budget': 0.05,
        },
        'runs': v1_records,
    }
    v1_manifest_path = (Path(DRIVE_RESULTS_DIR) / 'evaluations_v1' /
                        f"v1_manifest_{time.strftime('%Y%m%d_%H%M%S')}.json")
    v1_manifest_path.parent.mkdir(parents=True, exist_ok=True)
    v1_manifest_path.write_text(json.dumps(v1_manifest, indent=2))
    print('v1 evaluation matrix complete:', v1_manifest_path)
else:
    print('v1 evaluation matrix disabled.')


In [ ]:
#@title 18. Train entropy-regularized v2 task-only ensemble
V2_ENTROPY_CHECKPOINT_PATHS = [
    str(Path(DRIVE_RESULTS_DIR) / 'training_v2' / 'csup_0_entropy' / f'seed_{seed}' /
        f'minikandinsky-minikanddpl-v2-entropy-task-loss-csup-0-wc-0-seed-{seed}-end.pt')
    for seed in SEEDS
]
if RUN_V2_ENTROPY_TRAINING:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    entropy_records = []
    for seed, drive_checkpoint_raw in zip(SEEDS, V2_ENTROPY_CHECKPOINT_PATHS):
        drive_checkpoint = Path(drive_checkpoint_raw)
        if drive_checkpoint.is_file():
            print('Reusing completed entropy checkpoint:', drive_checkpoint)
            entropy_records.append({'seed': seed, 'checkpoint': str(drive_checkpoint), 'reused': True})
            continue
        log_path = run_job(
            'minikand_train', seed=seed, epochs=FULL_EPOCHS,
            batch_size=BATCH_SIZE, c_sup=0.0, w_c=0.0,
            checkpoint_tag='v2-entropy-task-loss', entropy=True, w_h=1.0,
        )
        local_checkpoint = (Path(REPO_DIR) / 'XOR_MNIST' / 'data' / 'ckpts' /
                            drive_checkpoint.name)
        if not local_checkpoint.is_file():
            raise FileNotFoundError(local_checkpoint)
        drive_checkpoint.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_checkpoint, drive_checkpoint)
        shutil.copy2(log_path, drive_checkpoint.parent / log_path.name)
        entropy_records.append({
            'seed': seed, 'checkpoint': str(drive_checkpoint),
            'log': log_path.name, 'reused': False,
        })
    entropy_manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': commit, 'dataset': 'minikandinsky',
        'model': 'minikanddpl', 'task': 'mini_patterns_bombazza',
        'concept_supervision': 0.0, 'concept_weight': 0.0,
        'entropy_regularization': True, 'entropy_weight': 1.0,
        'epochs': FULL_EPOCHS, 'batch_size': BATCH_SIZE, 'seeds': SEEDS,
        'dataset_sha256': sha256(Path(KAND_ZIP)), 'runs': entropy_records,
    }
    entropy_manifest_path = Path(DRIVE_RESULTS_DIR) / 'training_v2' / 'entropy_training_manifest.json'
    entropy_manifest_path.parent.mkdir(parents=True, exist_ok=True)
    entropy_manifest_path.write_text(json.dumps(entropy_manifest, indent=2))
    print('Entropy-control training complete:', entropy_manifest_path)
else:
    print('Entropy-control training disabled.')


In [ ]:
#@title 19. Validate v2 sweep checkpoints
if RUN_V2_REPRESENTATION_SWEEP:
    sweep_checkpoint_groups = (
        ('csup_1', V1_CSUP1_CHECKPOINT_PATHS),
        ('csup_0_entropy', V2_ENTROPY_CHECKPOINT_PATHS),
    )
    for condition, paths in sweep_checkpoint_groups:
        missing = [path for path in paths if not Path(path).is_file()]
        if missing:
            raise FileNotFoundError(f'Missing {condition} sweep checkpoints: ' + ', '.join(missing))
    print('Both v2 representation-sweep ensembles are ready.')
else:
    print('v2 representation sweep disabled.')


In [ ]:
#@title 20. Run validation-only representation sweep
if RUN_V2_REPRESENTATION_SWEEP:
    sweep_specs = (
        ('csup_1', V1_CSUP1_CHECKPOINT_PATHS, 1.0, 10.0, False),
        ('csup_0_entropy', V2_ENTROPY_CHECKPOINT_PATHS, 0.0, 0.0, True),
    )
    sweep_records = []
    for condition, checkpoints, c_sup, w_c, entropy in sweep_specs:
        run_name = f"{time.strftime('%Y%m%d_%H%M%S')}_{condition}"
        output_dir = Path(DRIVE_RESULTS_DIR) / 'representation_sweep_v2' / run_name
        command = [
            'python', 'colab_runner.py', '--job', 'minikand_representation_sweep',
            '--seed', '0', '--batch-size', str(BATCH_SIZE),
            '--minikand-output-dir', str(output_dir),
            '--minikand-c-sup', str(c_sup), '--minikand-w-c', str(w_c),
            '--minikand-checkpoints', *checkpoints,
        ]
        if entropy:
            command.extend(['--minikand-entropy', '--minikand-w-h', '1.0'])
        print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
        subprocess.run(command, cwd=REPO_DIR, check=True)
        selection_path = output_dir / 'representation_selection.json'
        selection = json.loads(selection_path.read_text())
        best = selection['best_observed_candidate']
        print(condition, 'selection status:', selection['selection_status'])
        print('Best:', best['representation_key'], best['normalization'],
              'AUROC=', best['auroc'], 'AP=', best['average_precision'])
        sweep_records.append({
            'condition': condition, 'output_directory': str(output_dir),
            'selection_status': selection['selection_status'],
            'selected_candidate': selection['selected_candidate'],
            'best_observed_candidate': best,
        })
    sweep_manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'test_split_evaluated': False, 'runs': sweep_records,
    }
    sweep_manifest_path = (Path(DRIVE_RESULTS_DIR) / 'representation_sweep_v2' /
                           f"sweep_manifest_{time.strftime('%Y%m%d_%H%M%S')}.json")
    sweep_manifest_path.parent.mkdir(parents=True, exist_ok=True)
    sweep_manifest_path.write_text(json.dumps(sweep_manifest, indent=2))
    print('Validation-only sweep complete:', sweep_manifest_path)
else:
    print('Validation-only representation sweep disabled.')


In [ ]:
#@title 21. Run fixed-representation v3 scoring sweep
if RUN_V3_SCORING_SWEEP:
    missing = [path for path in V1_CSUP1_CHECKPOINT_PATHS if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Missing supervised v1 checkpoints: ' + ', '.join(missing))
    run_name = time.strftime('%Y%m%d_%H%M%S') + '_csup_1'
    output_dir = Path(DRIVE_RESULTS_DIR) / 'scoring_sweep_v3' / run_name
    command = [
        'python', 'colab_runner.py', '--job', 'minikand_scoring_sweep',
        '--seed', '0', '--batch-size', str(BATCH_SIZE),
        '--minikand-output-dir', str(output_dir),
        '--minikand-c-sup', '1.0', '--minikand-w-c', '10.0',
        '--minikand-representation-key', 'CS',
        '--minikand-representation-normalization', 'zscore_l2',
        '--minikand-cross-fit-folds', '5', '--minikand-shrinkage', '0.10',
        '--minikand-checkpoints', *V1_CSUP1_CHECKPOINT_PATHS,
    ]
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)
    selection_path = output_dir / 'scoring_selection.json'
    selection = json.loads(selection_path.read_text())
    print('v3 selection status:', selection['selection_status'])
    for candidate in selection['candidates']:
        threshold = candidate['threshold_selection']
        print(candidate['scorer'], 'AUROC=', candidate['auroc'],
              'AP=', candidate['average_precision'], 'recall=', threshold['recall'],
              'false-review=', threshold['false_review_rate'],
              'accepted=', candidate['acceptance_criteria_satisfied'])
    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'test_split_evaluated': False, 'output_directory': str(output_dir),
        'selection_status': selection['selection_status'],
        'selected_candidate': selection['selected_candidate'],
        'best_observed_candidate': selection['best_observed_candidate'],
    }
    manifest_path = (Path(DRIVE_RESULTS_DIR) / 'scoring_sweep_v3' /
                     f"scoring_manifest_{time.strftime('%Y%m%d_%H%M%S')}.json")
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print('Validation-only v3 scoring sweep complete:', manifest_path)
else:
    print('v3 scoring sweep disabled.')


In [ ]:
#@title 22. Run frozen-candidate v4 uncertainty ablation
if RUN_V4_UNCERTAINTY_ABLATION:
    missing = [path for path in V1_CSUP1_CHECKPOINT_PATHS if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Missing supervised v1 checkpoints: ' + ', '.join(missing))
    run_name = time.strftime('%Y%m%d_%H%M%S') + '_csup_1'
    output_dir = Path(DRIVE_RESULTS_DIR) / 'uncertainty_ablation_v4' / run_name
    command = [
        'python', 'colab_runner.py', '--job', 'minikand_uncertainty_ablation',
        '--seed', '0', '--batch-size', str(BATCH_SIZE),
        '--minikand-output-dir', str(output_dir),
        '--minikand-c-sup', '1.0', '--minikand-w-c', '10.0',
        '--minikand-representation-key', 'CS',
        '--minikand-representation-normalization', 'zscore_l2',
        '--minikand-cross-fit-folds', '5', '--minikand-shrinkage', '0.10',
        '--minikand-checkpoints', *V1_CSUP1_CHECKPOINT_PATHS,
    ]
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)
    ablation_path = output_dir / 'uncertainty_ablation.json'
    ablation = json.loads(ablation_path.read_text())
    print('v4 status:', ablation['selection_status'])
    for candidate in ablation['candidates']:
        threshold = candidate['threshold_selection']
        print(candidate['scorer'], 'AUROC=', candidate['auroc'],
              'AP=', candidate['average_precision'], 'recall=', threshold['recall'],
              'false-review=', threshold['false_review_rate'])
    for comparison in ablation['baseline_comparison']:
        print('Fusion minus', comparison['baseline'], comparison)
    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'test_split_evaluated': False, 'output_directory': str(output_dir),
        'frozen_v3_candidate_result': ablation['frozen_v3_candidate_result'],
        'frozen_v3_reproduction_delta': ablation['frozen_v3_reproduction_delta'],
        'best_observed_candidate': ablation['best_observed_candidate'],
        'baseline_comparison': ablation['baseline_comparison'],
    }
    manifest_path = (Path(DRIVE_RESULTS_DIR) / 'uncertainty_ablation_v4' /
                     f"ablation_manifest_{time.strftime('%Y%m%d_%H%M%S')}.json")
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print('Validation-only v4 uncertainty ablation complete:', manifest_path)
else:
    print('v4 uncertainty ablation disabled.')


## Completion gate

A complete v0 run contains the three original supervised checkpoints plus `run_summary.json` artifacts for `figure_permute` and `palette_cycle`. A complete v1 comparison contains six newly trained task-loss checkpoints and four evaluation runs. The exploratory v2 stage adds three entropy-regularized task-only checkpoints and two validation-only representation selections. v3 keeps the supervised `CS + zscore_l2` representation fixed and compares four predeclared, cross-fitted risk scorers. v4 preserves the accepted v3 fusion and compares it with disagreement-only, predictive-entropy, and confidence-deficit controls. None of these development stages evaluates a test split; a new OOD test transform is frozen only after the validation controls are complete.
